# DS2002 · SQL Challenge Set

**Lab — 2026-09-11 · Fall 2026**  

---

## Lab 03 — SQL Challenge Set

Seven questions, one query each. Every query has to produce the right answer when the notebook is run from a fresh kernel, top to bottom.

Two rules that matter as much as getting the answer:

- **Check the row count** against what you expect before you believe a result.
- **Decide what to do about the untagged track and the unplayed tracks.** Several of these questions have a defensible answer either way; what is not defensible is not noticing they exist.

In [1]:
import sqlite3, pandas as pd
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE artists (artist_id INTEGER PRIMARY KEY, name TEXT, country TEXT);
CREATE TABLE tracks (track_id INTEGER PRIMARY KEY, title TEXT, artist_id INTEGER, genre TEXT, seconds INTEGER);
CREATE TABLE plays (play_id INTEGER PRIMARY KEY, track_id INTEGER, user TEXT, played_on TEXT);
INSERT INTO artists VALUES
 (1,'Nova Waves','US'),(2,'The Blue Ridge','US'),(3,'Kestrel','UK'),(4,'Marisol','ES');
INSERT INTO tracks VALUES
 (10,'Skyline',1,'Pop',201),(11,'Undertow',1,'Pop',240),(12,'Foothills',2,'Folk',185),
 (13,'Aurora',3,'Electronic',300),(14,'Nightfall',3,'Electronic',275),(15,'Sol',4,'Latin',210),
 (16,'Coastline',2,'Folk',199),(17,'Ridgeline',2,'Folk',225),(18,'Untitled Demo',3,NULL,150);
INSERT INTO plays VALUES
 (100,10,'ava','2026-09-01'),(101,10,'ben','2026-09-01'),(102,13,'ava','2026-09-02'),
 (103,13,'cara','2026-09-02'),(104,14,'ben','2026-09-03'),(105,12,'ava','2026-09-03'),
 (106,15,'dan','2026-09-04'),(107,10,'cara','2026-09-04'),(108,13,'dan','2026-09-05'),
 (109,16,'ava','2026-09-05'),(110,11,'ben','2026-09-06');
''')
conn.commit()

def q(sql):
    return pd.read_sql_query(sql, conn)
print('ready')

ready


### Q1 — Every track with its artist's name and country.

*Expected: 9 rows, one per track.*

In [2]:
q('''
SELECT t.title, a.name, a.country
FROM tracks t
JOIN artists a ON t.artist_id = a.artist_id
''')

,title,name,country
0,Skyline,Nova Waves,US
1,Undertow,Nova Waves,US
2,Foothills,The Blue Ridge,US
3,Aurora,Kestrel,UK
4,Nightfall,Kestrel,UK
5,Sol,Marisol,ES
6,Coastline,The Blue Ridge,US
7,Ridgeline,The Blue Ridge,US
8,Untitled Demo,Kestrel,UK


Since I need information from both the tracks and artists tables, I joined the tables using the shared artist_id column. Then I selected track title, artist name, and artist country from this new table.

### Q2 — Which genre has the longest average track length?

Return the genre and the average, not just the name.

In [3]:
q('''
SELECT genre, AVG(seconds) AS average_track_length
FROM tracks
WHERE genre IS NOT NULL
GROUP BY genre
ORDER BY average_track_length DESC
LIMIT 1
''')

,genre,average_track_length
0,Electronic,287.5


I grouped tracks by genre to create a row for each genre, displaying in two columns the genre's name and a new column for the average length of tracks in that genre. It selects only the rows where genre is not null, since there is a track with a null genre and I want to avoid the possibility of the null genre being the longest. I ordered the new table by average track length descending, so the genre with the highest average is on top, and then limited the table to 1 so only that row of the table is displayed.

### Q3 — For each user: how many plays, and how many distinct tracks?

Someone who played one track four times is a different listener from someone who played four different tracks. Your result should make that visible.

In [4]:
q('''
SELECT user, COUNT(*) AS plays, COUNT(DISTINCT track_id) AS distinct_tracks
FROM plays
GROUP BY user
''')

,user,plays,distinct_tracks
0,ava,4,4
1,ben,3,3
2,cara,2,2
3,dan,2,2


I looked at the plays table, grouping by user to display just one row per user name. During the grouping, COUNT(*) creates a new column that counts the amount of rows each user had in the original plays table, thus corresponding to each user's total plays. COUNT(DISTINCT track_id) does the same thing, but only counts the number of rows for each user that contained distinct track ids, thus corresponding to the number of distinct tracks each user played.

### Q4 — Which tracks have never been played?

*Expected: 2 rows.* Hint: `LEFT JOIN` and then keep the rows where the right side came back `NULL`.

In [5]:
q('''
SELECT t.title
FROM tracks t
LEFT JOIN plays p ON t.track_id = p.track_id
WHERE p.play_id IS NULL
''')

,title
0,Ridgeline
1,Untitled Demo


Since I want information from both the plays and tracks tables, I joined them using the shared track_id column. LEFT JOIN is necessary here since I am looking for tracks with no plays, therefore those tracks are not included in the plays table but I do not want them to be dropped. Then I filtered to only display those rows where play_id is Null.

### Q5 — Rank artists by total listening time.

Sum the seconds actually listened across all plays, most to least, and include a minutes column rounded to one decimal.

In [6]:
q('''
SELECT a.name, SUM(t.seconds) AS total_seconds, ROUND(SUM(t.seconds)/60, 1) AS total_minutes
FROM plays p
JOIN tracks t ON p.track_id = t.track_id
JOIN artists a ON t.artist_id = a.artist_id
GROUP BY a.name
ORDER BY total_seconds DESC
''')

,name,total_seconds,total_minutes
0,Kestrel,1175,19.0
1,Nova Waves,843,14.0
2,The Blue Ridge,384,6.0
3,Marisol,210,3.0


Since I wanted to use information from all of the data tables, I joined tracks and plays using the shared track_id column, and I joined artists using the column artist_id shared between artists and tracks. I grouped this table by artist name to create just one row for each artist, and during the grouping created a column for the sum of seconds for each artist and the sum of minutes by dividing seconds by 60. I ordered by total_seconds descending in order to rank the artists by total time played.

### Q6 — Which tracks are missing a genre?

Return the track id and title. Then, in a comment, say what `WHERE genre != 'Pop'` would have done to these rows and why.

In [7]:
q('''
SELECT track_id, title
FROM tracks
WHERE genre IS NULL
''')

,track_id,title
0,18,Untitled Demo


I selected just the track id and title columns from the tracks table, and filtered to only the tracks where genre is null.

If I had used WHERE genre != 'Pop', this row with the Null genre would have been ignored. Any other genre compared as non-equal to Pop evaluates as true, and would be printed, but comparing Null != 'Pop' evaluates as unknown, not true, and therefore the row would not be printed.

### Q7 — Plays per day.

`played_on` is stored as text like `'2026-09-01'`. Count plays per date, earliest first, and include the number of distinct users active that day.

In [8]:
q('''
SELECT played_on, COUNT(*) AS plays_per_date, COUNT(DISTINCT user) AS active_users
FROM plays
GROUP BY played_on
ORDER BY played_on ASC
''')

,played_on,plays_per_date,active_users
0,2026-09-01,2,2
1,2026-09-02,2,2
2,2026-09-03,2,2
3,2026-09-04,2,2
4,2026-09-05,2,2
5,2026-09-06,1,1


I grouped the plays table by date to create just one row for each date when a track was played, and ordered it in ascending order to show the earliest date first. During the grouping, I used COUNT(*) to count the amount of rows per date, corresponding to the total number of plays that occurred on that date. I also used COUNT(DISTINCT user) to count the number of rows with different users per date, corresponding to the number of active users.

### Validate your work

**TODO:** uncomment these and make them pass. Assign your query results to the variables as you go — for example `q1 = q('''...''')`.

In [9]:
assert len(q('''
SELECT t.title, a.name, a.country
FROM tracks t
JOIN artists a ON t.artist_id = a.artist_id
''')) == 9, 'Q1 should return one row per track'
assert len(q('''
SELECT t.title
FROM tracks t
LEFT JOIN plays p ON t.track_id = p.track_id
WHERE p.play_id IS NULL
''')) == 2, 'Q4: two tracks have never been played'
assert q('''
SELECT user, COUNT(*) AS plays, COUNT(DISTINCT track_id) AS distinct_tracks
FROM plays
GROUP BY user
''')['plays'].sum() == 11, 'Q3 should account for all 11 plays'
print('checks passed.')

checks passed.


### Write-up

Pick the query that gave you the most trouble and explain what you had wrong before you had it right. Name the specific misunderstanding — "I put the aggregate in WHERE" or "I used an inner join and lost the tracks with no plays" — not "it was confusing."

In Query 2, when selecting the genre with the longest average track length, I initially tried to SELECT genre, MAX(AVG(seconds)) and group by genre in order to return only the genre with the highest average track length. I ultimately realized I needed to create a new column for the average track length, and then it was easier to just sort in descending order and cut off all but the top row.